# 117 — Prompt, recurso, tool, skill, workflow y agente

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

Seis abstracciones con contrato distinto:

- **Prompt:** plantilla parametrizable; la elige el usuario/app; sin efectos.
- **Resource:** datos de solo lectura por URI; los inyecta la aplicación; sin efectos.
- **Tool:** función tipada invocada POR DECISIÓN DEL MODELO; única primitiva con
  efectos propios declarados.
- **Skill:** conocimiento procedimental empaquetado (instrucciones + ejemplos +
  scripts) que el modelo carga cuando la tarea coincide; sin efectos propios.
- **Workflow:** grafo de pasos escrito por el ingeniero; control de flujo del código.
- **Agente:** bucle donde el modelo decide tool, orden y parada; control del modelo.

### 🎛️ Los dos ejes que ordenan la taxonomía

```text
¿Quién decide su uso?   usuario/app: prompt, resource · ingeniero: workflow
                        modelo: tool, skill, agente
¿Puede causar efectos?  nunca: prompt, resource, skill · declarados: tool
                        heredados de sus tools: workflow, agente
```

Corolario de seguridad: los permisos (clase 119) se aplican sobre TOOLS — la única
primitiva con efectos propios; agentes y workflows heredan ese riesgo más el de
composición. Regla de selección: usar la abstracción más barata que resuelva —
prompt → resource → tool → skill → workflow → agente (y agente siempre con
presupuesto + permisos desde el día uno).

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** El resultado completo es un **agente**: `result.trace` muestra un bucle
acción→observación con parada por objetivo. Dentro aparecen **tools** (`status`, `sum`
con argumentos tipados en `action.args`). No aparecen: prompt (no hay plantilla
parametrizable), resource (no se inyectó dato externo por URI), skill (no hay
procedimiento empaquetado) ni workflow (el orden no está en un grafo externo — aunque
la política del lab sea determinista, el contrato es de agente).

**Ejercicio 2.** (a) prompt — usuario/app, sin efectos; (b) resource — aplicación, sin
efectos; (c) tool — modelo, efecto de escritura reversible; (d) skill — el modelo la
carga, sin efectos propios; (e) workflow — ingeniero, efectos de sus tools;
(f) agente — modelo, efectos de sus tools + composición; (g) tool — modelo, solo
lectura (pero con riesgo de entrada: contenido no confiable); (h) resource —
aplicación, sin efectos.

**Ejercicio 3.** (a) resource (la política como dato citable; un prompt solo no
garantiza vigencia); (b) workflow (pasos y orden conocidos: costo fijo, auditoría por
nodo); (c) agente (los pasos no se conocen a priori: hipótesis → consulta → nueva
hipótesis), con presupuesto; (d) prompt o workflow de un paso (tarea de una llamada:
lo más barato que funcione).

**Ejercicio 4.** Violaciones: (1) una skill NO tiene efectos propios — aquí "cargar la
skill" ejecuta un borrado, saltándose la frontera skill/tool; (2) el efecto
(irreversible en producción) no pasa por permisos ni aprobación. Diseño correcto: la
skill contiene el procedimiento (criterios de duplicado, orden de pasos, umbrales); el
borrado es una tool tipada `dedupe_rows(table, dry_run, idempotency_key)` clasificada
como escritura irreversible en la matriz de permisos (119), con dry-run obligatorio y
aprobación humana (120) antes de `dry_run=false`.

In [ ]:
result = run_lab("agent", seed=117)
assert result["kind"] == "agent"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — verificación sobre el JSON real
result = run_lab("agent", seed=117)
assert result["kind"] == "agent"
tools_usadas = [p["action"]["tool"] for p in result["result"]["trace"]]
print("abstraccion del conjunto: agente (bucle accion->observacion con parada)")
print("tools dentro del bucle:", tools_usadas)
assert tools_usadas == ["status", "sum"]
# ausentes: prompt, resource, skill, workflow (no hay plantilla, URI, paquete ni grafo)


In [ ]:
# Ejercicio 2 — clasificación de referencia
artefactos = {
    "a_plantilla_resumen": {"abstraccion": "prompt",   "controla": "usuario/app", "efectos": False},
    "b_schema_ventas":     {"abstraccion": "resource", "controla": "aplicacion",  "efectos": False},
    "c_create_ticket":     {"abstraccion": "tool",     "controla": "modelo",      "efectos": True},
    "d_cierre_contable":   {"abstraccion": "skill",    "controla": "modelo",      "efectos": False},
    "e_pipeline_fijo":     {"abstraccion": "workflow", "controla": "ingeniero",   "efectos": "de sus tools"},
    "f_bucle_depura_test": {"abstraccion": "agente",   "controla": "modelo",      "efectos": "de sus tools"},
    "g_search_web":        {"abstraccion": "tool",     "controla": "modelo",      "efectos": False},
    "h_historial_cliente": {"abstraccion": "resource", "controla": "aplicacion",  "efectos": False},
}
solo_con_efectos_propios = [k for k, v in artefactos.items() if v["efectos"] is True]
print("unica primitiva con efectos propios declarados:", solo_con_efectos_propios)


## Reflexión

1. En MCP los prompts son user-controlled, los resources application-controlled y las
   tools model-controlled. ¿Por qué ese reparto de control es una decisión de seguridad
   y qué se rompería si los resources fueran model-controlled con escritura?
2. Un "resource" de solo lectura no tiene efectos, pero ¿qué riesgo de ENTRADA
   introduce y con qué clase del OWASP LLM Top 10 se corresponde?
3. ¿Qué señal, medida sobre trayectorias reales (clase 122), justificaría degradar un
   agente a workflow, y por qué esa degradación es una mejora y no una pérdida?